In [53]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from collections import Counter

from datasets import load_dataset



In [54]:
# Load SMS Spam dataset
print('Loading SMS Spam dataset...')
sms_data = load_dataset('ucirvine/sms_spam', split='train')
dataset_split = sms_data.train_test_split(
    test_size=0.2, seed=42, stratify_by_column='label'
)
train_data = dataset_split['train']
test_data = dataset_split['test']
print(f'Training examples: {len(train_data)}')
print(f'Test examples: {len(test_data)}')


Loading SMS Spam dataset...
Training examples: 4459
Test examples: 1115


In [55]:
print(train_data)
print(train_data[0])


Dataset({
    features: ['sms', 'label'],
    num_rows: 4459
})
{'sms': 'Dont you have message offer\n', 'label': 0}


In [56]:
train_sentences = [example['sms'].split() for example in train_data]
train_labels = train_data['label']
test_sentences = [example['sms'].split() for example in test_data]
test_labels = test_data['label']
print(f'First training sentence: {train_sentences[0]}')

vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}

word_counts = Counter()

for sentence in train_sentences:
    word_counts.update(sentence)

for word, count in word_counts.items():
    if count >= 2:
        vocab[word] = len(vocab)

print(f'Vocabulary size: {len(vocab)}')
print(f'Sample vocab: {list(vocab.items())[:10]}')



First training sentence: ['Dont', 'you', 'have', 'message', 'offer']
Vocabulary size: 5353
Sample vocab: [('<PAD>', 0), ('<UNK>', 1), ('Dont', 2), ('you', 3), ('have', 4), ('message', 5), ('offer', 6), ('Ü', 7), ('all', 8), ('write', 9)]


In [57]:
import os
import gdown

file_id = "1WTbdKo7sNTfvRim2Sxm_aSNn2sCibe1q"
url = f"https://drive.google.com/uc?id={file_id}"
output = "glove_vectors.txt"

if not os.path.exists(output):
    print("Downloading GloVe vectors...")
    gdown.download(url, output, quiet=False)
else:
    print("GloVe vectors already downloaded. Skipping download.")

GloVe vectors already downloaded. Skipping download.


In [58]:
EMBEDDING_DIM = 100

glove_vectors = {}

with open("glove_vectors.txt", "r", encoding="utf-8") as f:
    for line in f:
        values = line.rstrip().split(" ")
        word = values[0]
        vector = np.array(values[1:], dtype="float32")

        if len(vector) == EMBEDDING_DIM:
            glove_vectors[word] = vector

print(f"Loaded {len(glove_vectors):,} GloVe vectors")

Loaded 1,291,147 GloVe vectors


In [59]:
embedding_matrix = np.random.randn(
    len(vocab),
    EMBEDDING_DIM
).astype("float32") * 0.01

for word, idx in vocab.items():

    if word == "<PAD>":
        embedding_matrix[idx] = np.zeros(
            EMBEDDING_DIM,
            dtype="float32"
        )

    elif word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        
        
print(f"Embedding matrix shape: {embedding_matrix.shape}")



Embedding matrix shape: (5353, 100)


In [60]:

class SMSDataset(Dataset):
    def __init__(self,messages, labels, vocab):
        self.messages = messages
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        message = self.messages[idx]
        label = self.labels[idx]

        # Convert message to indices
        message_indices = [
            self.vocab.get(word, self.vocab["<UNK>"]) for word in message
        ]
        if not message_indices:
            message_indices = [self.vocab['<UNK>']]

        return torch.tensor(message_indices, dtype=torch.long), torch.tensor(label, dtype=torch.long)

    

In [61]:
# Take several SMS messages of different lengths and pad them to the same length
def collate_fn(batch):
    
    messages, labels = zip(*batch)

    lengths = torch.tensor([len(message) for message in messages])

    padded_messages = pad_sequence(
        messages,
        batch_first=True,
        padding_value=vocab['<PAD>']
    )

    labels = torch.stack(labels)

    return padded_messages, labels, lengths

In [62]:
BATCH_SIZE = 32

train_dataset = SMSDataset(train_sentences, train_labels, vocab)
test_dataset = SMSDataset(test_sentences, test_labels, vocab)

train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)


In [65]:
class VanillaRNN(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False
        )
        self.rnn = nn.RNN(
            input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        return self.fc(hidden[-1])


class GRUModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False
        )
        self.gru = nn.GRU(
            input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        return self.fc(hidden[-1])


class LSTMModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.fc(hidden[-1])


In [66]:
RNN_HIDDEN_DIM = 128
OUTPUT_DIM = 2

MODEL_CLASSES = {
    "Vanilla RNN": VanillaRNN,
    "GRU": GRUModel,
    "LSTM": LSTMModel,
}


In [67]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
results = {}

for model_name, model_class in MODEL_CLASSES.items():
    # Keep initialization reproducible for each architecture.
    torch.manual_seed(42)
    model = model_class(
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=RNN_HIDDEN_DIM,
        output_dim=OUTPUT_DIM,
        embedding_matrix=embedding_matrix,
    ).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for messages, labels, lengths in train_dataloader:
            messages, labels = messages.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(messages)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)

        if (epoch + 1) % 10 == 0:
            epoch_loss = running_loss / len(train_dataset)
            print(f'{model_name} | Epoch {epoch + 1}/{NUM_EPOCHS} | loss: {epoch_loss:.4f}')

    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for messages, labels, lengths in test_dataloader:
            logits = model(messages.to(DEVICE))
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            targets.extend(labels.tolist())

    results[model_name] = {
        "accuracy": accuracy_score(targets, predictions),
        "spam_precision": precision_score(targets, predictions, pos_label=1, zero_division=0),
        "spam_recall": recall_score(targets, predictions, pos_label=1, zero_division=0),
        "spam_f1": f1_score(targets, predictions, pos_label=1, zero_division=0),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(targets, predictions),
        "report": classification_report(targets, predictions, target_names=["Ham", "Spam"], digits=4, zero_division=0),
    }

print(f'\nTest-set comparison (device: {DEVICE}):')
print(f'{"Model":<14} {"Accuracy":>10} {"Spam P":>10} {"Spam R":>10} {"Spam F1":>10} {"Macro F1":>10}')
for model_name, metrics in results.items():
    print(
        f'{model_name:<14} {metrics["accuracy"]:>10.4f} '
        f'{metrics["spam_precision"]:>10.4f} {metrics["spam_recall"]:>10.4f} '
        f'{metrics["spam_f1"]:>10.4f} {metrics["macro_f1"]:>10.4f}'
    )

best_model = max(results, key=lambda name: results[name]["spam_f1"])
print(f'\nBest spam F1: {best_model} ({results[best_model]["spam_f1"]:.4f})')
for model_name, metrics in results.items():
    print(f'\n{model_name} confusion matrix (rows=true [Ham, Spam], columns=predicted [Ham, Spam]):')
    print(metrics["confusion_matrix"])
    print(metrics["report"])


Vanilla RNN | Epoch 10/100 | loss: 0.3739
Vanilla RNN | Epoch 20/100 | loss: 0.3628
Vanilla RNN | Epoch 30/100 | loss: 0.3130
Vanilla RNN | Epoch 40/100 | loss: 0.3893
Vanilla RNN | Epoch 50/100 | loss: 0.3830
Vanilla RNN | Epoch 60/100 | loss: 0.1209
Vanilla RNN | Epoch 70/100 | loss: 0.1045
Vanilla RNN | Epoch 80/100 | loss: 0.3761
Vanilla RNN | Epoch 90/100 | loss: 0.0836
Vanilla RNN | Epoch 100/100 | loss: 0.2689
GRU | Epoch 10/100 | loss: 0.0537
GRU | Epoch 20/100 | loss: 0.0039
GRU | Epoch 30/100 | loss: 0.0025
GRU | Epoch 40/100 | loss: 0.0021
GRU | Epoch 50/100 | loss: 0.0016
GRU | Epoch 60/100 | loss: 0.0016
GRU | Epoch 70/100 | loss: 0.0016
GRU | Epoch 80/100 | loss: 0.0022
GRU | Epoch 90/100 | loss: 0.0016
GRU | Epoch 100/100 | loss: 0.0016
LSTM | Epoch 10/100 | loss: 0.0478
LSTM | Epoch 20/100 | loss: 0.1340
LSTM | Epoch 30/100 | loss: 0.0181
LSTM | Epoch 40/100 | loss: 0.0406
LSTM | Epoch 50/100 | loss: 0.0240
LSTM | Epoch 60/100 | loss: 0.0043
LSTM | Epoch 70/100 | loss: 